# CNDS — Interactive Lab

Notebook client for the **Cognitive Network Defense System** API. It talks to a
running CNDS instance (`uvicorn src.api.main:app` or the Docker stack) over plain
HTTP/WebSocket — it does not need to run on the same machine as the API.

**Sections**
1. Setup & connection
2. Engine / system health
3. Unsupervised baseline subsystem
4. Feature-vector lab (craft a synthetic flow → `/api/predict`)
5. Alerts explorer
6. Trends dashboard
7. Incidents
8. Live alert stream (WebSocket)
9. Export
10. Advanced: live traffic generator (Scapy, real packets — needs root/CAP_NET_RAW)

> Designed to be general-purpose: every endpoint exposed by `src/api/` is reachable
> from a widget below. Sections that need elevated privileges (10) degrade gracefully
> with a clear error if the permission isn't available on the host running the kernel.


## 1. Setup

Install notebook-only dependencies (kept out of `requirements.txt` since the API
itself doesn't need them). Safe to re-run.


In [ ]:
%pip install -q ipywidgets plotly pandas requests websockets scapy


In [ ]:
import asyncio
import json
import time
from datetime import datetime, timezone

import ipywidgets as W
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import requests
from IPython.display import display, clear_output, FileLink

# ── Connection config ───────────────────────────────────────────────────────
api_url_w   = W.Text(value="http://localhost:8000", description="API URL", layout=W.Layout(width="400px"))
api_key_w   = W.Password(value="", description="X-API-Key", layout=W.Layout(width="400px"))
jwt_token_w = W.Password(value="", description="Bearer JWT", layout=W.Layout(width="400px"))

display(W.VBox([
    W.HTML("<b>Connection</b> — leave API key / JWT blank if the server has auth disabled."),
    api_url_w, api_key_w, jwt_token_w,
]))


def base_url() -> str:
    return api_url_w.value.rstrip("/")


def headers() -> dict:
    h = {}
    if api_key_w.value:
        h["X-API-Key"] = api_key_w.value
    if jwt_token_w.value:
        h["Authorization"] = f"Bearer {jwt_token_w.value}"
    return h


def api_get(path, **params):
    r = requests.get(f"{base_url()}{path}", headers=headers(), params=params, timeout=10)
    r.raise_for_status()
    return r.json()


def api_post(path, payload):
    r = requests.post(f"{base_url()}{path}", headers=headers(), json=payload, timeout=10)
    r.raise_for_status()
    return r.json()


def api_patch(path, payload):
    r = requests.patch(f"{base_url()}{path}", headers=headers(), json=payload, timeout=10)
    r.raise_for_status()
    return r.json()


## 2. Engine / system health

Hits `GET /health`. Run after changing the connection settings above.

In [ ]:
health_out = W.Output()

def refresh_health(_=None):
    with health_out:
        clear_output()
        try:
            h = requests.get(f"{base_url()}/health", timeout=5).json()
        except Exception as e:
            print(f"Could not reach API at {base_url()}: {e}")
            return
        rows = [{"engine": k, "active": v} for k, v in h.get("engines", {}).items()]
        df = pd.DataFrame(rows)
        display(W.HTML(f"<b>status:</b> {h.get('status')}"))
        display(df.style.map(lambda v: "background-color:#c8e6c9" if v is True
                              else ("background-color:#ffcdd2" if v is False else "")))
        if h.get("capture_stats"):
            display(W.HTML("<b>capture_stats:</b>"))
            display(pd.DataFrame([h["capture_stats"]]))

health_btn = W.Button(description="Refresh health", button_style="info")
health_btn.on_click(refresh_health)
display(W.VBox([health_btn, health_out]))
refresh_health()


## 3. Unsupervised baseline subsystem

`GET /api/baseline/status` — collector progress toward the auto-training trigger
(50k vectors / 20 distinct IPs / 30 min / 2.5-bit port entropy, whichever is the
bottleneck), the currently loaded baseline model, and drift vs. the last trained
window. `GET /api/baseline/windows` — history of past training runs.


In [ ]:
baseline_out = W.Output()

def refresh_baseline(_=None):
    with baseline_out:
        clear_output()
        try:
            status = api_get("/api/baseline/status")
        except Exception as e:
            print(f"baseline/status failed: {e}")
            return

        collector = status.get("collector")
        trigger = status.get("trigger")
        if collector and trigger:
            display(W.HTML(f"<b>Collector</b> — {collector['n_total']} vectors, "
                            f"{collector['distinct_src_ips']} distinct src IPs, "
                            f"{collector['elapsed_sec']}s elapsed, "
                            f"port entropy {collector['dst_port_entropy_bits']} bits "
                            f"{'(training in flight)' if collector['training_in_flight'] else ''}"))
            bars = []
            for cond, ratio in trigger["progress_ratios"].items():
                bar = W.FloatProgress(value=min(ratio, 1.0), min=0, max=1.0, description=cond,
                                       bar_style="success" if ratio >= 1 else "info",
                                       layout=W.Layout(width="450px"))
                bars.append(bar)
            display(W.VBox(bars))
            eta = trigger.get("eta_estimate_sec")
            display(W.HTML(f"<b>Bottleneck:</b> {trigger['bottleneck']} · "
                            f"<b>ETA:</b> {eta if eta is not None else 'n/a'}s"))
        else:
            display(W.HTML("<i>Collector not active (BASELINE_COLLECTION_ENABLED=false?)</i>"))

        engine = status.get("engine", {})
        display(W.HTML(f"<b>Loaded baseline model:</b> {json.dumps(engine)}"))

        drift = status.get("drift")
        if drift:
            display(W.HTML(f"<b>Drift vs last window:</b> {json.dumps(drift)}"))

        try:
            windows = api_get("/api/baseline/windows", limit=10)["windows"]
            if windows:
                display(W.HTML("<b>Recent training windows:</b>"))
                display(pd.json_normalize(windows))
        except Exception as e:
            print(f"baseline/windows failed: {e}")

baseline_btn = W.Button(description="Refresh baseline", button_style="info")
baseline_btn.on_click(refresh_baseline)
display(W.VBox([baseline_btn, baseline_out]))
refresh_baseline()


## 4. Feature-vector lab

Craft a synthetic detection request and send it straight to `POST /api/predict` —
the same call the live capture pipeline makes after flow expiry. All 76 flow
features and 18 host features are exposed by name (`FLOW_FEATURE_NAMES` /
`HOST_FEATURE_NAMES`, taken from `src/features/flow_extractor.py` and
`host_extractor.py`) so this lab works for *any* scenario, not just the presets.


In [ ]:
FLOW_FEATURE_NAMES = [
    "flow_duration", "tot_fwd_pkts", "tot_bwd_pkts",
    "totlen_fwd_pkts", "totlen_bwd_pkts",
    "fwd_pkt_len_max", "fwd_pkt_len_min", "fwd_pkt_len_mean", "fwd_pkt_len_std",
    "bwd_pkt_len_max", "bwd_pkt_len_min", "bwd_pkt_len_mean", "bwd_pkt_len_std",
    "flow_byts_s", "flow_pkts_s",
    "flow_iat_mean", "flow_iat_std", "flow_iat_max", "flow_iat_min",
    "fwd_iat_tot", "fwd_iat_mean", "fwd_iat_std", "fwd_iat_max", "fwd_iat_min",
    "bwd_iat_tot", "bwd_iat_mean", "bwd_iat_std", "bwd_iat_max", "bwd_iat_min",
    "fwd_psh_flags", "bwd_psh_flags", "fwd_urg_flags", "bwd_urg_flags",
    "fwd_header_len", "bwd_header_len",
    "fwd_pkts_s", "bwd_pkts_s",
    "pkt_len_min", "pkt_len_max", "pkt_len_mean", "pkt_len_std", "pkt_len_var",
    "fin_flag_cnt", "syn_flag_cnt", "rst_flag_cnt", "psh_flag_cnt",
    "ack_flag_cnt", "urg_flag_cnt", "cwr_flag_count", "ece_flag_cnt",
    "down_up_ratio", "pkt_size_avg",
    "fwd_seg_size_avg", "bwd_seg_size_avg",
    "fwd_byts_b_avg", "fwd_pkts_b_avg", "fwd_blk_rate_avg",
    "bwd_byts_b_avg", "bwd_pkts_b_avg", "bwd_blk_rate_avg",
    "subflow_fwd_pkts", "subflow_fwd_byts",
    "subflow_bwd_pkts", "subflow_bwd_byts",
    "init_fwd_win_byts", "init_bwd_win_byts",
    "fwd_act_data_pkts", "fwd_seg_size_min",
    "active_mean", "active_std", "active_max", "active_min",
    "idle_mean", "idle_std", "idle_max", "idle_min",
]
assert len(FLOW_FEATURE_NAMES) == 76

HOST_FEATURE_NAMES = [
    "packets_per_sec", "bytes_per_sec", "avg_packet_size", "packet_size_var",
    "total_packets", "total_bytes",
    "iat_mean", "iat_std", "burst_rate", "session_duration",
    "tcp_ratio", "udp_ratio", "icmp_ratio",
    "unique_ports", "uncommon_port_ratio",
    "avg_payload_entropy", "avg_payload_size", "payload_size_var",
]
assert len(HOST_FEATURE_NAMES) == 18

PATTERN_NAMES = ["sql_injection", "xss", "command_injection", "path_traversal", "log4j", "shellshock"]

# Preset scenarios: sparse overrides on top of an all-zero vector.
PRESETS = {
    "benign": {"flow": {"tot_fwd_pkts": 10, "tot_bwd_pkts": 10, "flow_pkts_s": 5, "flow_duration": 2.0},
               "host": {"packets_per_sec": 5, "bytes_per_sec": 2000, "tcp_ratio": 1.0, "unique_ports": 1},
               "payload": []},
    "dos_syn_flood": {"flow": {"syn_flag_cnt": 5000, "tot_fwd_pkts": 5000, "tot_bwd_pkts": 0,
                                "flow_pkts_s": 50000, "flow_duration": 0.1, "pkt_len_mean": 60},
                       "host": {"packets_per_sec": 50000, "burst_rate": 0.98, "tcp_ratio": 1.0,
                                "unique_ports": 1, "uncommon_port_ratio": 0.0},
                       "payload": []},
    "port_scan": {"flow": {"tot_fwd_pkts": 1, "tot_bwd_pkts": 0, "syn_flag_cnt": 1, "flow_duration": 0.01},
                  "host": {"unique_ports": 800, "uncommon_port_ratio": 0.95, "packets_per_sec": 900,
                           "tcp_ratio": 1.0},
                  "payload": []},
    "sqli_web": {"flow": {"tot_fwd_pkts": 3, "tot_bwd_pkts": 2, "fwd_pkt_len_mean": 400, "flow_duration": 0.3},
                 "host": {"avg_payload_entropy": 4.2, "avg_payload_size": 400, "tcp_ratio": 1.0,
                          "unique_ports": 1},
                 "payload": ["sql_injection"]},
    "data_exfil": {"flow": {"tot_fwd_pkts": 2000, "tot_bwd_pkts": 50, "totlen_fwd_pkts": 50_000_000,
                             "down_up_ratio": 0.02, "flow_duration": 60},
                   "host": {"bytes_per_sec": 2_000_000, "avg_payload_entropy": 7.8,
                             "avg_payload_size": 1400, "tcp_ratio": 1.0},
                   "payload": ["large_payload", "asymmetric_upload"]},
}

print(f"{len(FLOW_FEATURE_NAMES)} flow features, {len(HOST_FEATURE_NAMES)} host features, "
      f"{len(PATTERN_NAMES)} payload patterns, {len(PRESETS)} presets loaded.")


In [ ]:
# ── Widgets: connection 5-tuple + preset picker ─────────────────────────────
src_ip_w  = W.Text(value="10.0.0.1", description="src_ip")
dst_ip_w  = W.Text(value="192.168.1.60", description="dst_ip")
src_port_w = W.IntText(value=55000, description="src_port")
dst_port_w = W.IntText(value=443, description="dst_port")
proto_w   = W.Dropdown(options=[("TCP", 6), ("UDP", 17), ("ICMP", 1)], value=6, description="protocol")

preset_w  = W.Dropdown(options=["(none)"] + list(PRESETS.keys()), value="(none)", description="Preset")
apply_preset_btn = W.Button(description="Apply preset", button_style="warning")

flow_sliders = {name: W.FloatText(value=0.0, description=name, style={"description_width": "160px"})
                for name in FLOW_FEATURE_NAMES}
host_sliders = {name: W.FloatText(value=0.0, description=name, style={"description_width": "160px"})
                for name in HOST_FEATURE_NAMES}
payload_multiselect = W.SelectMultiple(options=PATTERN_NAMES, description="payload_matches",
                                        layout=W.Layout(width="300px"))

def apply_preset(_=None):
    name = preset_w.value
    for w in flow_sliders.values():
        w.value = 0.0
    for w in host_sliders.values():
        w.value = 0.0
    payload_multiselect.value = ()
    if name == "(none)":
        return
    p = PRESETS[name]
    for k, v in p["flow"].items():
        flow_sliders[k].value = v
    for k, v in p["host"].items():
        host_sliders[k].value = v
    payload_multiselect.value = tuple(x for x in p["payload"] if x in PATTERN_NAMES)

apply_preset_btn.on_click(apply_preset)

flow_accordion = W.Accordion(children=[W.GridBox(list(flow_sliders.values()),
                              layout=W.Layout(grid_template_columns="repeat(2, 320px)"))])
flow_accordion.set_title(0, f"76 flow features (FLOW_FEATURE_NAMES)")
host_accordion = W.Accordion(children=[W.GridBox(list(host_sliders.values()),
                              layout=W.Layout(grid_template_columns="repeat(2, 320px)"))])
host_accordion.set_title(0, f"18 host features (HOST_FEATURE_NAMES)")

predict_btn = W.Button(description="Send to /api/predict", button_style="danger")
predict_out = W.Output()

display(W.VBox([
    W.HBox([src_ip_w, dst_ip_w, src_port_w, dst_port_w, proto_w]),
    W.HBox([preset_w, apply_preset_btn]),
    flow_accordion,
    host_accordion,
    payload_multiselect,
    predict_btn,
    predict_out,
]))


In [ ]:
def render_predict_response(resp: dict):
    es = resp.get("engine_scores", {}) or {}
    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=resp["ensemble_score"],
        title={"text": f"Ensemble score — {resp.get('severity','?').upper()}"},
        gauge={"axis": {"range": [0, 1]},
               "bar": {"color": "crimson" if resp["is_anomaly"] else "seagreen"}},
    ))
    fig.update_layout(height=250, margin=dict(t=40, b=0, l=0, r=0))
    fig.show()

    engine_names = ["supervised", "isolation_forest", "lstm", "rules"]
    vals = [es.get(n) for n in engine_names]
    bar = px.bar(x=engine_names, y=[v if v is not None else 0 for v in vals],
                 labels={"x": "engine", "y": "score"}, title="Per-engine scores (None → 0, engine inactive)")
    bar.update_layout(height=300)
    bar.show()

    print(f"attack_type:     {resp.get('attack_type')}")
    print(f"active_engines:  {resp.get('active_engines')}")
    print(f"triggered_rules: {es.get('triggered_rules')}")
    print(f"alert_id:        {resp.get('alert_id')}  (None ⇒ below threshold / deduped / suppressed)")
    if resp.get("mitre_techniques"):
        display(pd.DataFrame(resp["mitre_techniques"]))
    if resp.get("src_geo"):
        print(f"src_geo: {resp['src_geo']}")


def send_predict(_=None):
    with predict_out:
        clear_output()
        payload = {
            "src_ip": src_ip_w.value,
            "dst_ip": dst_ip_w.value or None,
            "src_port": src_port_w.value,
            "dst_port": dst_port_w.value,
            "protocol": proto_w.value,
            "flow_features": [flow_sliders[n].value for n in FLOW_FEATURE_NAMES],
            "host_features": [host_sliders[n].value for n in HOST_FEATURE_NAMES],
            "payload_matches": list(payload_multiselect.value),
        }
        try:
            resp = api_post("/api/predict", payload)
        except Exception as e:
            print(f"predict failed: {e}")
            return
        render_predict_response(resp)

predict_btn.on_click(send_predict)


## 5. Alerts explorer

`GET /api/alerts` with filters, plus inline acknowledge via `PATCH /api/alerts/{id}`.

In [ ]:
sev_w   = W.SelectMultiple(options=["low", "medium", "high", "critical"], description="severity")
ack_w   = W.Dropdown(options=[("any", None), ("unacknowledged", False), ("acknowledged", True)],
                      description="ack")
srcip_w = W.Text(value="", description="src_ip")
limit_w = W.IntSlider(value=50, min=5, max=500, step=5, description="limit")

alerts_out = W.Output()
alerts_df_holder = {"df": pd.DataFrame()}

def fetch_alerts(_=None):
    with alerts_out:
        clear_output()
        params = {"limit": limit_w.value}
        if ack_w.value is not None:
            params["acknowledged"] = ack_w.value
        if srcip_w.value:
            params["src_ip"] = srcip_w.value
        try:
            if len(sev_w.value) == 1:
                params["severity"] = sev_w.value[0]
            alerts = api_get("/api/alerts", **params)
        except Exception as e:
            print(f"alerts fetch failed: {e}")
            return
        if sev_w.value and len(sev_w.value) > 1:
            alerts = [a for a in alerts if a["severity"] in sev_w.value]
        df = pd.json_normalize(alerts)
        alerts_df_holder["df"] = df
        if df.empty:
            print("No alerts match the current filters.")
            return
        cols = [c for c in ["id", "timestamp", "src_ip", "dst_ip", "attack_type",
                             "severity", "ensemble_score", "acknowledged"] if c in df.columns]
        display(df[cols])

fetch_btn = W.Button(description="Fetch alerts", button_style="info")
fetch_btn.on_click(fetch_alerts)

ack_id_w = W.IntText(description="alert_id")
ack_val_w = W.Checkbox(value=True, description="acknowledged")
notes_w = W.Text(description="notes")
ack_btn = W.Button(description="Apply PATCH", button_style="success")
ack_out = W.Output()

def do_ack(_=None):
    with ack_out:
        clear_output()
        body = {"acknowledged": ack_val_w.value}
        if notes_w.value:
            body["notes"] = notes_w.value
        try:
            resp = api_patch(f"/api/alerts/{ack_id_w.value}", body)
            print("Updated:", resp)
        except Exception as e:
            print(f"patch failed: {e}")

ack_btn.on_click(do_ack)

display(W.VBox([
    W.HBox([sev_w, ack_w, srcip_w]),
    limit_w, fetch_btn, alerts_out,
    W.HTML("<b>Acknowledge / annotate an alert</b>"),
    W.HBox([ack_id_w, ack_val_w, notes_w, ack_btn]), ack_out,
]))
fetch_alerts()


## 6. Trends dashboard

`GET /api/alerts/trends` bucketed by hour or day, stacked by severity.

In [ ]:
trend_hours_w  = W.IntSlider(value=24, min=1, max=168, step=1, description="hours")
trend_bucket_w = W.Dropdown(options=["hour", "day"], value="hour", description="bucket")
trend_btn = W.Button(description="Refresh trends", button_style="info")
trend_out = W.Output()

def refresh_trends(_=None):
    with trend_out:
        clear_output()
        try:
            data = api_get("/api/alerts/trends", hours=trend_hours_w.value, bucket=trend_bucket_w.value)
        except Exception as e:
            print(f"trends fetch failed: {e}")
            return
        buckets = data["data"]
        if not buckets:
            print("No alerts in this window.")
            return
        rows = []
        for ts, info in sorted(buckets.items()):
            row = {"bucket": ts, "total": info["total"]}
            row.update(info["by_severity"])
            rows.append(row)
        df = pd.DataFrame(rows).fillna(0)
        sev_cols = [c for c in ["low", "medium", "high", "critical"] if c in df.columns]
        fig = px.bar(df, x="bucket", y=sev_cols, title=f"Alerts per {data['bucket']} (last {data['hours']}h)",
                     labels={"value": "alerts", "bucket": "time"})
        fig.show()

trend_btn.on_click(refresh_trends)
display(W.VBox([W.HBox([trend_hours_w, trend_bucket_w]), trend_btn, trend_out]))
refresh_trends()


## 7. Incidents

`GET /api/incidents` and `POST /api/incidents` (requires `admin`/`analyst` role if JWT auth is enabled).

In [ ]:
inc_status_w = W.Dropdown(options=[("any", None), ("open", "open"), ("closed", "closed")], description="status")
inc_list_btn = W.Button(description="List incidents", button_style="info")
inc_out = W.Output()

def list_incidents(_=None):
    with inc_out:
        clear_output()
        params = {} if inc_status_w.value is None else {"status": inc_status_w.value}
        try:
            incidents = api_get("/api/incidents", **params)
        except Exception as e:
            print(f"incidents fetch failed: {e}")
            return
        display(pd.json_normalize(incidents) if incidents else "No incidents.")

inc_list_btn.on_click(list_incidents)

inc_title_w = W.Text(description="title")
inc_desc_w  = W.Text(description="description")
inc_sev_w   = W.Dropdown(options=["low", "medium", "high", "critical"], description="severity")
inc_create_btn = W.Button(description="Create incident", button_style="success")
inc_create_out = W.Output()

def create_incident(_=None):
    with inc_create_out:
        clear_output()
        body = {"title": inc_title_w.value, "description": inc_desc_w.value, "severity": inc_sev_w.value}
        try:
            resp = api_post("/api/incidents", body)
            print("Created:", resp)
        except Exception as e:
            print(f"create failed: {e}")

inc_create_btn.on_click(create_incident)

display(W.VBox([
    W.HBox([inc_status_w, inc_list_btn]), inc_out,
    W.HTML("<b>Create incident</b>"),
    W.HBox([inc_title_w, inc_desc_w, inc_sev_w, inc_create_btn]), inc_create_out,
]))


## 8. Live alert stream (WebSocket)

Connects to `ws://.../ws/alerts` and appends every broadcast alert to a live table
as it fires — e.g. while you replay scenarios in section 4, or while the real
capture pipeline (`main.py`) is running elsewhere against this same API/DB.

Requires `ipykernel>=6` (autoawait is on by default in modern Jupyter/JupyterLab).


In [ ]:
import websockets

ws_rows = []
ws_task_holder = {"task": None}
ws_start_btn = W.Button(description="Start stream", button_style="success")
ws_stop_btn  = W.Button(description="Stop stream", button_style="danger")
ws_live_out  = W.Output()

def _ws_url():
    url = base_url().replace("http://", "ws://").replace("https://", "wss://")
    url += "/ws/alerts"
    if jwt_token_w.value:
        url += f"?token={jwt_token_w.value}"
    return url

async def _stream_alerts():
    url = _ws_url()
    with ws_live_out:
        print(f"Connecting to {url} ...")
    try:
        async with websockets.connect(url) as ws:
            with ws_live_out:
                print("Connected. Waiting for alerts...")
            async for message in ws:
                try:
                    data = json.loads(message)
                except json.JSONDecodeError:
                    continue
                data["_received_at"] = datetime.now(timezone.utc).isoformat()
                ws_rows.append(data)
                with ws_live_out:
                    clear_output()
                    df = pd.json_normalize(ws_rows[-25:])
                    cols = [c for c in ["_received_at", "src_ip", "dst_ip", "attack_type",
                                         "severity", "ensemble_score"] if c in df.columns]
                    print(f"{len(ws_rows)} alert(s) received this session — showing last 25:")
                    display(df[cols] if cols else df)
    except asyncio.CancelledError:
        with ws_live_out:
            print("Stream stopped.")
        raise
    except Exception as e:
        with ws_live_out:
            print(f"WebSocket error: {e}")

def start_stream(_=None):
    if ws_task_holder["task"] is not None and not ws_task_holder["task"].done():
        with ws_live_out:
            print("Already running.")
        return
    ws_task_holder["task"] = asyncio.create_task(_stream_alerts())

def stop_stream(_=None):
    task = ws_task_holder["task"]
    if task is not None and not task.done():
        task.cancel()

ws_start_btn.on_click(start_stream)
ws_stop_btn.on_click(stop_stream)
display(W.VBox([W.HBox([ws_start_btn, ws_stop_btn]), ws_live_out]))


## 9. Export

`GET /api/alerts/export` (JSON or CSV) — requires `admin`/`analyst` role if JWT auth is enabled.

In [ ]:
export_fmt_w = W.Dropdown(options=["json", "csv"], description="format")
export_hours_w = W.IntText(value=24, description="hours (blank=all)")
export_btn = W.Button(description="Export alerts", button_style="info")
export_out = W.Output()

def do_export(_=None):
    with export_out:
        clear_output()
        params = {"format": export_fmt_w.value}
        if export_hours_w.value:
            params["hours"] = export_hours_w.value
        try:
            r = requests.get(f"{base_url()}/api/alerts/export", headers=headers(), params=params, timeout=15)
            r.raise_for_status()
        except Exception as e:
            print(f"export failed: {e}")
            return
        fname = f"cnds_alerts_export.{export_fmt_w.value}"
        with open(fname, "wb") as f:
            f.write(r.content)
        print(f"Saved {len(r.content)} bytes to {fname}")
        display(FileLink(fname))

export_btn.on_click(do_export)
display(W.VBox([W.HBox([export_fmt_w, export_hours_w]), export_btn, export_out]))


## 10. Advanced — live traffic generator (Scapy, real packets)

Crafts and **sends real packets on the wire** so the full pipeline gets exercised
(Scapy capture → dispatcher → feature extractors → ensemble), not just the
`/api/predict` shortcut from section 4. Use this against a host where CNDS's
`main.py` (or `docker-compose` `detector` service) is actively capturing.

**Requirements (not guaranteed on every machine — that's expected, not a bug):**
- Run the kernel as root, or grant the Python process `CAP_NET_RAW`/`CAP_NET_ADMIN`
  (`sudo setcap cap_net_raw,cap_net_admin+eip $(readlink -f $(which python3))`).
- `target_iface` must match `CAPTURE_INTERFACE` of the running detector.
- Sending floods/scans at anything other than your own lab network is out of scope —
  only point this at infrastructure you own or are authorized to test.


In [ ]:
from scapy.all import IP, TCP, UDP, ICMP, Raw, send, sendp, RandShort, conf as scapy_conf

target_ip_w   = W.Text(value="192.168.1.60", description="target_ip")
target_port_w = W.IntText(value=443, description="target_port")
iface_w       = W.Text(value=str(scapy_conf.iface) if scapy_conf.iface else "", description="iface")
attack_w      = W.Dropdown(
    options=["syn_flood", "icmp_flood", "port_scan", "sqli_http", "xss_http", "large_upload"],
    description="attack")
count_w       = W.IntSlider(value=50, min=1, max=2000, step=1, description="packet count")
rate_w        = W.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.01, description="delay (s)")
fire_btn      = W.Button(description="Fire (sends REAL packets)", button_style="danger")
fire_out      = W.Output()


def _build_packets():
    tip, tport = target_ip_w.value, target_port_w.value
    kind = attack_w.value
    if kind == "syn_flood":
        return [IP(dst=tip) / TCP(sport=int(RandShort()), dport=tport, flags="S") for _ in range(count_w.value)]
    if kind == "icmp_flood":
        return [IP(dst=tip) / ICMP() for _ in range(count_w.value)]
    if kind == "port_scan":
        return [IP(dst=tip) / TCP(sport=int(RandShort()), dport=p, flags="S")
                for p in range(1, count_w.value + 1)]
    if kind == "sqli_http":
        payload = b"GET /login?user=' OR '1'='1';-- HTTP/1.1\r\nHost: target\r\n\r\n"
        return [IP(dst=tip) / TCP(sport=int(RandShort()), dport=tport, flags="PA") / Raw(load=payload)]
    if kind == "xss_http":
        payload = b"GET /search?q=<script>alert(1)</script> HTTP/1.1\r\nHost: target\r\n\r\n"
        return [IP(dst=tip) / TCP(sport=int(RandShort()), dport=tport, flags="PA") / Raw(load=payload)]
    if kind == "large_upload":
        return [IP(dst=tip) / TCP(sport=int(RandShort()), dport=tport, flags="PA") / Raw(load=b"A" * 1400)
                for _ in range(count_w.value)]
    raise ValueError(kind)


def fire(_=None):
    with fire_out:
        clear_output()
        try:
            packets = _build_packets()
        except Exception as e:
            print(f"Failed to build packets: {e}")
            return
        print(f"Sending {len(packets)} packet(s) for scenario '{attack_w.value}' "
              f"to {target_ip_w.value} via iface={iface_w.value or '(default)'} ...")
        try:
            for pkt in packets:
                send(pkt, iface=iface_w.value or None, verbose=False)
                if rate_w.value:
                    time.sleep(rate_w.value)
            print("Done. Check the live alert stream (section 8) or the dashboard for detections.")
        except PermissionError as e:
            print(f"Permission denied — CAP_NET_RAW not available to this kernel: {e}")
        except Exception as e:
            print(f"Send failed: {e}")


fire_btn.on_click(fire)
display(W.VBox([
    W.HBox([target_ip_w, target_port_w, iface_w]),
    W.HBox([attack_w, count_w, rate_w]),
    fire_btn, fire_out,
]))


## Troubleshooting

- **`requests.exceptions.ConnectionError`** — check `api_url_w` and that the API
  process is running (`uvicorn src.api.main:app`).
- **401 on every call** — `API_KEY` or `JWT_SECRET` is set server-side; fill in
  section 1's `X-API-Key` / `Bearer JWT` fields.
- **WebSocket connects then immediately closes with code 4001/4003** — JWT auth is
  enabled and the token is missing/expired; refresh it in section 1.
- **Section 10 `PermissionError`** — the kernel process lacks `CAP_NET_RAW`; either
  run Jupyter as root or `setcap` the interpreter as noted above. This is a host
  permission issue, not a notebook bug — every other section works over plain HTTP
  regardless of capture privileges.
